In [26]:
import plotly.graph_objects as go

def twh_to_ej(twh):
    """
    Convert Terawatt-hours (TWh) to Exajoules (EJ).

    Parameters:
    twh (float): Energy in Terawatt-hours.

    Returns:
    float: Energy in Exajoules.
    """
    conversion_factor = 0.0036
    return twh * conversion_factor

def xy(d):
    return zip(*sorted(d.items())) if d else ([], [])

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_CCS_const_value_ep.xml`
    * `/input/policy/korea-2035/power/coal_CCS_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_const_value_ep.xml`
    * `/input/policy/korea-2035/power/coal_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_turnoff.xml`
    * `/input/policy/korea-2035/power/coal_shutdown_cp.xml`
    * `/input/policy/korea-2035/power/coal_shutdown_ep.xml`

# Coal

Coal power generations (TWh) are projected for 2023, 2030, and 2035 in BPESD. The generation for 2025 is calculated by linear interpolation. These generations are prjected as the figure below.

In [37]:
dictCapTWh = {2020: 198.1, 2023: 188.5, 2025: 154.237, 2030: 110.5, 2035: 88.9}

In [38]:
dictCapTWhEp = {2020: 198.1, 2023: 188.5, 2025: 154.237}
for year in range(2030, 2040, 5):
    dictCapTWhEp[year] = max(dictCapTWhEp[2025] * (1 - (year - 2025) / (2035 - 2025)), 0)
dictCapTWhEp

{2020: 198.1, 2023: 188.5, 2025: 154.237, 2030: 77.1185, 2035: 0.0}

In [39]:
years_cap, values_cap = xy(dictCapTWh)
years_alt, values_alt = xy(dictCapTWhEp)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
    ("Enhanced Ambition", years_alt, values_alt, "dash"),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2023, 2030, 2035]
annotations = []
for d in (dictCapTWh, dictCapTWhEp):
    for yr in target_years:
        val = d.get(yr)
        if val is not None:
            annotations.append(go.layout.Annotation(
                x=yr, y=val,
                xanchor='center', yanchor='bottom',
                text=f"{val:.1f} TWh",
                showarrow=True, arrowhead=1, ax=0, ay=-20
            ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(title='Year', title_font=dict(size=18), tickfont=dict(size=15)),
    yaxis=dict(title='TWh',  title_font=dict(size=18), tickfont=dict(size=15)),
)

fig.write_image("../figures/coal_generation.png", scale=2)
fig.show()

However in the *Current policies* scenario, these projected generations include ammonia co-firing generations. We calculated ammonia co-firing generation first and subtract this amount from total coal generations to implement ceilings for coal power.

The 11th BPESD projects total hydrogen + ammonia power generation at 15.5 TWh in 2030, 32.8 TWh in 2035, and 43.9 TWh in 2038. However, it does not specify the individual generation amounts for hydrogen and ammonia. In contrast, the 10th BPESD projected hydrogen power generation of 6.1 TWh and ammonia power generation of 6.9 TWh in 2030.

In this study, it is assumed that the annual ratio of hydrogen to ammonia generation in the 11th Basic Plan is the same as the ratio in 2030 under the 10th Basic Plan. Accordingly, the hydrogen generation share is $\frac{6.1}{13} = 46.9\%$, and the ammonia generation share is $53.1\%$.

Annual “traditional” coal-fired power generation is defined as coal generation excluding ammonia co-firing. In 2030 and 2035, the total coal–ammonia co-firing generation (20% ammonia co-firing) amounts to:

$$15.5×0.531×5,32.8×0.531×5$$

which correspond to 41.15 TWh and 87.08 TWh, respectively, for 2030 and 2035. Since only 80% of this generation is attributable to coal, we calculate:

$$41.15×0.8,87.08×0.8$$

resulting in 32.9 TWh and 69.7 TWh. These values are then subtracted from the 2030 and 2035 coal generation figures in the 11th Basic Plan to obtain coal generation without co-firing.

In [25]:
print(f"year = 2030, ammonia co-firing (TWh) = {twh_to_ej(41.15):.3f}")
print(f"year = 2035, ammonia co-firing (TWh) = {twh_to_ej(87.08):.3f}")

year = 2030, ammonia co-firing (TWh) = 0.148
year = 2035, ammonia co-firing (TWh) = 0.313


As a by product we get ceilings for ammonia co-firing for *Current Policieis* scenario, `/input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml`

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Coal-Ammonia-Blend-Ceiling">
        <policyType>tax</policyType>
        <market>South Korea</market>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2030">0.148</constraint>
        <constraint year="2035">0.313</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>


```

We now subtract co-firing generations from total coal power generations to implement ceilings for conventional coal powers: `/input/policy/korea-2035/power/coal_const_value_cp.xml`

In [16]:
dictCapTWhRe = {2020: 198.1, 2023: 188.5, 2025: 154.237, 2030: 110.5 - 32.9, 2035: 88.9 - 69.7}

In [17]:
print(f"year = 2020, conv. coal (EJ) = {twh_to_ej(dictCapTWhRe[2020]):.3f}")
print(f"year = 2025, conv. coal (EJ) = {twh_to_ej(dictCapTWhRe[2025]):.3f}")
print(f"year = 2030, conv. coal (EJ) = {twh_to_ej(dictCapTWhRe[2030]):.3f}")
print(f"year = 2035, conv. coal (EJ) = {twh_to_ej(dictCapTWhRe[2035]):.3f}")

year = 2020, conv. coal (EJ) = 0.713
year = 2025, conv. coal (EJ) = 0.555
year = 2030, conv. coal (EJ) = 0.279
year = 2035, conv. coal (EJ) = 0.069


```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Coal-Generation-Ceiling">
        <policyType>tax</policyType>
        <market>South Korea</market>
        <min-price year="2020">0</min-price>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.713</constraint>
        <constraint year="2025">0.555</constraint>
        <constraint year="2030">0.279</constraint>
        <constraint year="2035">0.069</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```

To implement ceilings for conventional coal powers in the *Enhanced Ambition* scenario: `/input/policy/korea-2035/power/coal_const_value_ep.xml`

In [18]:
print(f"year = 2020, conv. coal (EJ) = {twh_to_ej(dictCapTWhEp[2020]):.3f}")
print(f"year = 2025, conv. coal (EJ) = {twh_to_ej(dictCapTWhEp[2025]):.3f}")
print(f"year = 2030, conv. coal (EJ) = {twh_to_ej(dictCapTWhEp[2030]):.3f}")
print(f"year = 2035, conv. coal (EJ) = {twh_to_ej(dictCapTWhEp[2035]):.3f}")

year = 2020, conv. coal (EJ) = 0.713
year = 2025, conv. coal (EJ) = 0.555
year = 2030, conv. coal (EJ) = 0.278
year = 2035, conv. coal (EJ) = 0.000


```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Coal-Generation-Ceiling">
        <policyType>tax</policyType>
        <market>South Korea</market>
        <min-price year="2020">0</min-price>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.713</constraint>
        <constraint year="2025">0.555</constraint>
        <constraint year="2030">0.277</constraint>
        <constraint year="2035">0.001</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```

Additionally we assumed that 

* there are no additional coal power capacities after 2025: `/input/policy/korea-2035/power/coal_turnoff.xml`
* the capacity factors are adjusted to meet the generation ceiling: `/input/policy/korea-2035/power/coal_shutdown_cp.xml`, `/input/policy/korea-2035/power/coal_shutdown_ep.xml`